# NB52 — Generation Protection Theorem and Profinite Structure

NB50–NB51 proved that the Gen1/Gen2 palindrome degeneracy survives all **real** cross-section couplings (via $\sigma_7$ time reversal) and is **diagonal-protected** for couplings that preserve the $k_7$ quantum number.

This notebook establishes a **stronger** and **final** result: the **combined time reversal** $\Sigma = \bigotimes_p \sigma_p$ is an exact symmetry of the full physical solenoid Hamiltonian — kinetic, potential, and gauge sectors — making Gen1 = Gen2 an **exact degeneracy** for all angular dynamics.

We also explore the **covering tower** (level-2 branching) and **profinite spectrum** (Cantor-set fiber), demonstrating that the palindrome persists at all covering levels with self-similar spectral structure.

### Identities established:
- **#79**: Momentum Anti-Palindrome — $\hat{p}(k) = -\hat{p}(n-k)$ on every $C_{p-1}$
- **#80**: Combined Time-Reversal Protection — $\Sigma = \bigotimes_p \sigma_p$ is an exact symmetry of all physical solenoid couplings
- **#81**: Generation Protection Theorem — $\Sigma$-invariance guarantees $E(\text{Gen1}) = E(\text{Gen2})$ for all physical couplings
- **#82**: Covering Tower Palindrome Persistence — all level-2 sub-states pairwise degenerate between Gen1 and Gen2
- **#83**: Profinite Spectral Self-Similarity — gap ratio = covering degree = 7

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
from numpy.linalg import eigh, eigvalsh
from solenoid_algebra import SA

primes = SA.primes
n2, n3, n5, n7 = 1, 2, 4, 6
N = n2 * n3 * n5 * n7  # 48

def cycle_laplacian(m):
    L = np.zeros((m, m))
    for i in range(m):
        L[i, i] = 2; L[i, (i+1)%m] = -1; L[i, (i-1)%m] = -1
    return L

def fourier_matrix(m):
    return np.array([[np.exp(2j*np.pi*j*k/m)/np.sqrt(m)
                      for k in range(m)] for j in range(m)])

def momentum_operator(m):
    # p = -i(T+ - T-)/2
    p = np.zeros((m, m), dtype=complex)
    for a in range(m):
        p[a, (a+1)%m] = -0.5j; p[a, (a-1)%m] = +0.5j
    return p

def time_reversal_matrix(m):
    # sigma: a -> -a mod m
    S = np.zeros((m, m))
    for a in range(m):
        S[(-a)%m, a] = 1.0
    return S

def lam(p, k):
    if p == 2: return 0.0
    return 2.0 - 2.0 * np.cos(2 * np.pi * k / (p - 1))

# Build operators
L7 = cycle_laplacian(n7); p7 = momentum_operator(n7); F7 = fourier_matrix(n7)
I2 = np.eye(n2); I3 = np.eye(n3); I5 = np.eye(n5); I7 = np.eye(n7)
F_full = np.kron(np.kron(np.kron(fourier_matrix(n2), fourier_matrix(n3)),
                          fourier_matrix(n5)), F7)

H0 = (np.kron(np.kron(np.kron(cycle_laplacian(n2), I3), I5), I7) +
      np.kron(np.kron(np.kron(I2, cycle_laplacian(n3)), I5), I7) +
      np.kron(np.kron(np.kron(I2, I3), cycle_laplacian(n5)), I7) +
      np.kron(np.kron(np.kron(I2, I3), I5), L7))

Sigma = np.kron(np.kron(np.kron(time_reversal_matrix(n2),
                                 time_reversal_matrix(n3)),
                         time_reversal_matrix(n5)),
                time_reversal_matrix(n7))

print(f"Cross-section: {n2} x {n3} x {n5} x {n7} = {N} states")
print(f"Sigma^2 = I: {np.allclose(Sigma @ Sigma, np.eye(N))}")
print(f"[H0, Sigma] = 0: {np.allclose(H0 @ Sigma, Sigma @ H0)}")

Cross-section: 1 x 2 x 4 x 6 = 48 states
Sigma^2 = I: True
[H0, Sigma] = 0: True


## 1. Momentum Anti-Palindrome (#79)

The **momentum operator** on $C_n$ is $\hat{p} = -i(T^+ - T^-)/2$, with Fourier eigenvalues $\hat{p}(k) = -\sin(2\pi k/n)$.

Unlike the Laplacian eigenvalues $\lambda(k) = 2 - 2\cos(2\pi k/n)$ which are **palindromic** ($\lambda(k) = \lambda(n-k)$), the momentum eigenvalues are **anti-palindromic**: $\hat{p}(k) = -\hat{p}(n-k)$.

This anti-palindromic structure is the origin of the gauge coupling's ability to *distinguish* Gen1 from Gen2 — though the combined time-reversal symmetry prevents this distinction from producing physical mass splitting.

In [2]:
# Momentum operator on C_6 in Fourier basis
p7_F = F7.conj().T @ p7 @ F7

print("Momentum eigenvalues on C_6:")
print(f"  {'k7':>3} {'p(k7)':>10} {'p(6-k7)':>10} {'sum':>10} {'palindrome?':>12}")
print("  " + "-" * 50)
all_anti = True
for k in range(n7):
    pk = p7_F[k, k].real
    pkp = p7_F[(n7-k)%n7, (n7-k)%n7].real
    is_anti = abs(pk + pkp) < 1e-10
    if not is_anti and k != (n7-k)%n7:
        all_anti = False
    status = "SELF" if k == (n7-k)%n7 else ("ANTI" if is_anti else "FAIL")
    print(f"  {k:>3} {pk:>10.6f} {pkp:>10.6f} {pk+pkp:>10.6f} {status:>12}")

print(f"\nAll non-self pairs anti-palindromic: {all_anti}")
print(f"p_hat_7 Hermitian: {np.allclose(p7, p7.conj().T)}")
print(f"p_hat_7 purely imaginary: {np.allclose(p7.real, 0)}")

# Compare with Laplacian (palindromic)
print(f"\nLaplacian eigenvalues (palindromic): {[round(lam(7,k), 1) for k in range(n7)]}")
print(f"Momentum eigenvalues (anti-palindromic): {[round(p7_F[k,k].real, 4) for k in range(n7)]}")

Momentum eigenvalues on C_6:
   k7      p(k7)    p(6-k7)        sum  palindrome?
  --------------------------------------------------
    0   0.000000   0.000000   0.000000         SELF
    1   0.866025  -0.866025   0.000000         ANTI
    2   0.866025  -0.866025   0.000000         ANTI
    3  -0.000000  -0.000000  -0.000000         SELF
    4  -0.866025   0.866025   0.000000         ANTI
    5  -0.866025   0.866025   0.000000         ANTI

All non-self pairs anti-palindromic: True
p_hat_7 Hermitian: True
p_hat_7 purely imaginary: True

Laplacian eigenvalues (palindromic): [np.float64(0.0), np.float64(1.0), np.float64(3.0), np.float64(4.0), np.float64(3.0), np.float64(1.0)]
Momentum eigenvalues (anti-palindromic): [np.float64(0.0), np.float64(0.866), np.float64(0.866), np.float64(-0.0), np.float64(-0.866), np.float64(-0.866)]


## 2. Combined Time-Reversal Symmetry (#80)

Define the **combined time reversal** $\Sigma = \bigotimes_p \sigma_p$, where $\sigma_p: a_p \to -a_p \bmod (p-1)$.

**Every** physical solenoid coupling is $\Sigma$-invariant:
- **Kinetic** $\hat{p}_k^2$: $(\text{odd})^2 = \text{even}$ ✓
- **Potential** $\cos(p\theta_{k-1} - \theta_k)$: $\cos$ is even ✓
- **Gauge** $\sin(\theta_{k-1}) \cdot \hat{p}_k$: $(\text{odd})(\text{odd}) = \text{even}$ ✓

The crucial point: the gauge coupling is **complex** in the site basis (purely imaginary), yet $\Sigma$-invariant because both factors are odd.

In [3]:
# Physical couplings
A_sin = np.array([np.sin(np.pi * a / 2) for a in range(n5)])  # sin(theta5)

# Gauge coupling: sin(theta5) x p_hat_7
V_gauge = np.kron(np.kron(np.kron(I2, I3), np.diag(A_sin)), p7)

# Phase coupling: cos(7*theta5 - theta7)
V_phase = np.zeros((N, N))
for a2 in range(n2):
    for a3 in range(n3):
        for a5 in range(n5):
            for a7 in range(n7):
                idx = ((a2*n3+a3)*n5+a5)*n7+a7
                V_phase[idx, idx] = np.cos(7*2*np.pi*a5/n5 - 2*np.pi*a7/n7)

print("Sigma-invariance of physical couplings:")
for name, V in [("H0 (kinetic)", H0),
                ("cos(7t5-t7) (potential)", V_phase),
                ("sin(t5)*p7 (gauge)", V_gauge)]:
    comm = np.max(np.abs(Sigma @ V @ Sigma.T - V))
    print(f"  {name:>30}: |[Sigma, V]| = {comm:.2e}  {'EVEN' if comm < 1e-10 else 'ODD'}")

# Verify gauge coupling properties
print(f"\nGauge coupling structure:")
print(f"  Hermitian: {np.allclose(V_gauge, V_gauge.conj().T)}")
print(f"  Purely imaginary in site basis: {np.allclose(V_gauge.real, 0)}")
print(f"  sin(theta5) T-parity under sigma5: T-odd")
print(f"  p_hat_7 T-parity under sigma7: T-odd")
print(f"  Product: (T-odd)(T-odd) = T-EVEN => Sigma-invariant")

Sigma-invariance of physical couplings:
                    H0 (kinetic): |[Sigma, V]| = 0.00e+00  EVEN
         cos(7t5-t7) (potential): |[Sigma, V]| = 1.22e-15  EVEN
              sin(t5)*p7 (gauge): |[Sigma, V]| = 1.22e-16  EVEN

Gauge coupling structure:
  Hermitian: True
  Purely imaginary in site basis: True
  sin(theta5) T-parity under sigma5: T-odd
  p_hat_7 T-parity under sigma7: T-odd
  Product: (T-odd)(T-odd) = T-EVEN => Sigma-invariant


## 3. Generation Protection Theorem (#81)

**Theorem.** *For any Hamiltonian $H$ on the $(2,3,5,7)$-solenoid cross-section that commutes with $\Sigma = \bigotimes_p \sigma_p$, the generation palindrome holds: $E(\text{Gen1}) = E(\text{Gen2})$.*

**Proof.** $\Sigma$ maps $k_7 \to n_7 - k_7$, which sends Gen1 $\leftrightarrow$ Gen2. For any eigenstate $\psi$ with eigenvalue $E$, $\Sigma\psi$ is also an eigenstate with the same $E$. Since $\Sigma$ bijectively maps Gen1 states to Gen2 states, the eigenvalue multisets are identical. $\square$

We verify this for the **full physical Hamiltonian** including both potential and gauge couplings.

In [4]:
# H = H0 + eta_p * V_phase + eta_g * V_gauge (PHYSICAL, Sigma-invariant)
print("Physical Hamiltonian: H = H0 + eta_p * cos(7t5-t7) + eta_g * sin(t5)*p7")
print(f"\n{'eta_p':>6} {'eta_g':>6} {'max|Gen1-Gen2| z2=0':>22} {'max|Gen1-Gen2| z2=1':>22}")
print("-" * 60)

all_pass = True
for eta_p in [0.5, 1.0, 2.0]:
    for eta_g in [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]:
        H = H0 + eta_p * V_phase + eta_g * V_gauge
        evals, evecs = eigh(H)
        overlap = np.abs(F_full.conj().T @ evecs)**2

        splits = {}
        for k7_pair, label in [((4, 2), "z0"), ((1, 5), "z1")]:
            k7_g1, k7_g2 = k7_pair
            e_g1, e_g2 = [], []
            for k3 in range(n3):
                for k5 in range(n5):
                    fi1 = ((0*n3+k3)*n5+k5)*n7+k7_g1
                    fi2 = ((0*n3+k3)*n5+k5)*n7+k7_g2
                    e_g1.append(evals[np.argmax(overlap[fi1, :])])
                    e_g2.append(evals[np.argmax(overlap[fi2, :])])
            e_g1, e_g2 = np.sort(e_g1), np.sort(e_g2)
            splits[label] = np.max(np.abs(e_g1 - e_g2))
            if splits[label] > 1e-8:
                all_pass = False

        print(f"{eta_p:>6.1f} {eta_g:>6.1f} {splits['z0']:>22.2e} {splits['z1']:>22.2e}")

print(f"\nAll palindromes preserved: {all_pass}")

Physical Hamiltonian: H = H0 + eta_p * cos(7t5-t7) + eta_g * sin(t5)*p7

 eta_p  eta_g    max|Gen1-Gen2| z2=0    max|Gen1-Gen2| z2=1
------------------------------------------------------------
   0.5    0.0               0.00e+00               0.00e+00
   0.5    0.1               0.00e+00               0.00e+00
   0.5    0.5               0.00e+00               0.00e+00
   0.5    1.0               0.00e+00               0.00e+00
   0.5    2.0               0.00e+00               0.00e+00
   0.5    5.0               0.00e+00               0.00e+00
   1.0    0.0               0.00e+00               0.00e+00
   1.0    0.1               0.00e+00               0.00e+00
   1.0    0.5               0.00e+00               0.00e+00
   1.0    1.0               0.00e+00               0.00e+00
   1.0    2.0               0.00e+00               0.00e+00
   1.0    5.0               0.00e+00               0.00e+00
   2.0    0.0               0.00e+00               0.00e+00
   2.0    0.1             

### Direct Sigma-pairing verification

For the physical Hamiltonian at strong coupling, verify that every eigenvalue has a $\Sigma$-partner at exactly the same energy.

In [5]:
# Strong coupling test
eta_p, eta_g = 1.5, 1.0
H = H0 + eta_p * V_phase + eta_g * V_gauge
print(f"H = H0 + {eta_p}*V_phase + {eta_g}*V_gauge")
print(f"[H, Sigma] = {np.max(np.abs(Sigma @ H @ Sigma.T - H)):.2e}")

evals, evecs = eigh(H)
max_dE = 0.0
for n in range(N):
    sigma_psi = Sigma @ evecs[:, n]
    overlaps = np.abs(evecs.conj().T @ sigma_psi)
    partner = np.argmax(overlaps)
    dE = abs(evals[n] - evals[partner])
    max_dE = max(max_dE, dE)

print(f"Max |E(n) - E(Sigma*n)| across all 48 states: {max_dE:.2e}")
print(f"All Sigma-paired: {max_dE < 1e-10}")

H = H0 + 1.5*V_phase + 1.0*V_gauge
[H, Sigma] = 2.66e-15
Max |E(n) - E(Sigma*n)| across all 48 states: 0.00e+00
All Sigma-paired: True


### Control: $\Sigma$-breaking coupling

A coupling that is **odd** under $\Sigma$ *does* break the palindrome. Example: $\sin(\theta_5) \otimes L_7$ — $\sin$ is $\sigma_5$-odd while $L_7$ is $\sigma_7$-even, making the product $\Sigma$-odd. This is **not** from the minimal coupling Lagrangian but demonstrates the mechanism needed for generation splitting.

In [6]:
# Non-physical Sigma-breaking coupling: sin(theta5) x L7
V_fwd = np.kron(np.kron(np.kron(I2, I3), np.diag(A_sin)), L7)
comm = np.max(np.abs(Sigma @ V_fwd @ Sigma.T - V_fwd))
print(f"V_fwd = sin(theta5) x L7: |[Sigma, V]| = {comm:.1f} (Sigma-ODD)")

# Combined Sigma-breaking Hamiltonian
eta_f, eta_g = 0.5, 0.1
H_break = H0 + eta_f * V_fwd + eta_g * V_gauge
H_F = F_full.conj().T @ H_break @ F_full

# Block comparison (both V_fwd and V_gauge are k7-diagonal in Fourier)
idx4 = [i for i in range(N) if i%n7 == 4]
idx2 = [i for i in range(N) if i%n7 == 2]
e4 = np.sort(eigvalsh(H_F[np.ix_(idx4, idx4)]))
e2 = np.sort(eigvalsh(H_F[np.ix_(idx2, idx2)]))
split = np.max(np.abs(e4 - e2))

print(f"\nSigma-breaking: H = H0 + {eta_f}*V_fwd + {eta_g}*V_gauge")
print(f"  Gen1(k7=4) vs Gen2(k7=2) block eigenvalue split: {split:.6f}")
print(f"  Palindrome: {'PRESERVED' if split < 1e-8 else 'BROKEN'}")

# Verify Sigma pairing fails
evals_b, evecs_b = eigh(H_break)
unpaired = sum(1 for n in range(N)
               if abs(evals_b[n] - evals_b[np.argmax(np.abs(evecs_b.conj().T @ (Sigma @ evecs_b[:, n])))]) > 1e-6)
print(f"  Eigenvalues NOT Sigma-paired: {unpaired}/{N}")

V_fwd = sin(theta5) x L7: |[Sigma, V]| = 4.0 (Sigma-ODD)

Sigma-breaking: H = H0 + 0.5*V_fwd + 0.1*V_gauge
  Gen1(k7=4) vs Gen2(k7=2) block eigenvalue split: 0.103883
  Palindrome: BROKEN
  Eigenvalues NOT Sigma-paired: 25/48


## 4. Covering Tower Palindrome Persistence (#82)

At level 2, the $p=7$ factor becomes $C_{42}$ (branching factor 7). The palindrome $k_7' \leftrightarrow 42 - k_7'$ maps Gen1 $\leftrightarrow$ Gen2 at every sub-level. Each level-1 generation branches into 7 sub-states, all pairwise degenerate between generations.

In [7]:
n7_L2 = 42  # C_42 at level 2

# Show that palindrome maps Gen1 <-> Gen2
gen1_L2 = [k for k in range(n7_L2) if (k%6)%3 == 1]
gen2_L2 = [k for k in range(n7_L2) if (k%6)%3 == 2]

all_degenerate = True
print("Level-2 palindrome Gen1 <-> Gen2 pairing:")
print(f"  {'Gen1 k7':>8} {'lam_L2':>10} {'Gen2 k7':>8} {'lam_L2':>10} {'|diff|':>10}")
for k in gen1_L2:
    partner = (n7_L2 - k) % n7_L2
    lk = 2 - 2*np.cos(2*np.pi*k/n7_L2)
    lp = 2 - 2*np.cos(2*np.pi*partner/n7_L2)
    degen = abs(lk - lp) < 1e-12
    if not degen: all_degenerate = False
    pgen = (partner%6)%3
    print(f"  {k:>8} {lk:>10.6f} {partner:>8} {lp:>10.6f} {abs(lk-lp):>10.2e}  gen={pgen}")

print(f"\nAll Gen1<->Gen2 pairs degenerate: {all_degenerate}")
print(f"Gen1 sub-states: {len(gen1_L2)}, Gen2 sub-states: {len(gen2_L2)}")

# Intra-generation fine structure (7 sub-levels per gen)
print(f"\nIntra-generation fine structure (Gen1, z2=1, k7=1):")
sub = sorted([k for k in range(n7_L2) if k%6 == 1])
lams = [2 - 2*np.cos(2*np.pi*k/n7_L2) for k in sub]
lam_min = min(l for l in lams if l > 1e-10)
for k, l in zip(sub, lams):
    print(f"  k7'={k:>2}: lambda = {l:.6f}, ratio to min = {l/lam_min:.2f}")

Level-2 palindrome Gen1 <-> Gen2 pairing:
   Gen1 k7     lam_L2  Gen2 k7     lam_L2     |diff|
         1   0.022338       41   0.022338   2.22e-16  gen=2
         4   0.347522       38   0.347522   8.88e-16  gen=2
         7   1.000000       35   1.000000   1.55e-15  gen=2
        10   1.850540       32   1.850540   0.00e+00  gen=2
        13   2.730682       29   2.730682   4.44e-16  gen=2
        16   3.466104       26   3.466104   0.00e+00  gen=2
        19   3.911146       23   3.911146   4.44e-16  gen=2
        22   3.977662       20   3.977662   4.44e-16  gen=2
        25   3.652478       17   3.652478   0.00e+00  gen=2
        28   3.000000       14   3.000000   1.33e-15  gen=2
        31   2.149460       11   2.149460   1.33e-15  gen=2
        34   1.269318        8   1.269318   6.66e-16  gen=2
        37   0.533896        5   0.533896   4.44e-16  gen=2
        40   0.088854        2   0.088854   2.22e-16  gen=2

All Gen1<->Gen2 pairs degenerate: True
Gen1 sub-states: 14, Gen2

## 5. Profinite Spectral Self-Similarity (#83)

At level $L$, the $p=7$ factor becomes $C_{7^L \cdot 6}$. The eigenvalue spectrum fills $[0, 4]$ as $L \to \infty$, with self-similar gap structure where the **gap ratio equals the covering degree** $p = 7$.

In [8]:
print("Profinite spectral approximation (p=7 factor):")
print(f"  {'Level':>5} {'Cycle':>8} {'Unique':>7} {'Min gap':>12} {'Max gap':>12} {'Max|G1-G2|':>12}")
print("  " + "-" * 60)

prev_max_gap = None
for L in range(1, 5):
    m = (7**L) * 6
    evals = np.array([2 - 2*np.cos(2*np.pi*k/m) for k in range(m)])
    evals_sorted = np.sort(evals)
    gaps = np.diff(evals_sorted)
    n_unique = len(np.unique(np.round(evals_sorted, 10)))
    min_gap = np.min(gaps[gaps > 1e-12]) if np.any(gaps > 1e-12) else 0
    max_gap = np.max(gaps)

    # Gen1/Gen2 degeneracy at this level
    gen1 = np.sort([2-2*np.cos(2*np.pi*k/m) for k in range(m) if (k%6)%3==1])
    gen2 = np.sort([2-2*np.cos(2*np.pi*k/m) for k in range(m) if (k%6)%3==2])
    max_g12 = np.max(np.abs(gen1 - gen2))

    print(f"  {L:>5} C_{m:<6} {n_unique:>7} {min_gap:>12.8f} {max_gap:>12.8f} {max_g12:>12.2e}")

# Gap scaling
print(f"\nGap ratio between levels:")
for L in range(1, 4):
    m1, m2 = (7**L)*6, (7**(L+1))*6
    g1 = np.max(np.diff(np.sort([2-2*np.cos(2*np.pi*k/m1) for k in range(m1)])))
    g2 = np.max(np.diff(np.sort([2-2*np.cos(2*np.pi*k/m2) for k in range(m2)])))
    ratio = g1 / g2
    print(f"  Level {L} -> {L+1}: max gap ratio = {ratio:.4f} (expected: 7)")

Profinite spectral approximation (p=7 factor):
  Level    Cycle  Unique      Min gap      Max gap   Max|G1-G2|
  ------------------------------------------------------------
      1 C_42          22   0.02233835   0.29892037     1.55e-15
      2 C_294        148   0.00045672   0.04274194     1.33e-15
      3 C_2058      1030   0.00000932   0.00610611     2.22e-15
      4 C_14406     7204   0.00000019   0.00087230     2.66e-15

Gap ratio between levels:
  Level 1 -> 2: max gap ratio = 6.9936 (expected: 7)
  Level 2 -> 3: max gap ratio = 6.9999 (expected: 7)
  Level 3 -> 4: max gap ratio = 7.0000 (expected: 7)


## Scorecard

In [9]:
print("NB52 SCORECARD")
print("=" * 65)
print(f"{'#':>4} {'Identity':>45} {'Status':>8}")
print("-" * 65)
identities = [
    (79, "Momentum Anti-Palindrome", "PASS"),
    (80, "Combined Time-Reversal Protection", "PASS"),
    (81, "Generation Protection Theorem", "PASS"),
    (82, "Covering Tower Palindrome Persistence", "PASS"),
    (83, "Profinite Spectral Self-Similarity", "PASS"),
]
for num, name, status in identities:
    print(f"{num:>4}  {name:>45}  {status:>8}")
print("-" * 65)
print(f"Running total: 83 predictions/identities, 0 free parameters")
print(f"Notebooks: 52 (NB01-NB52)")

NB52 SCORECARD
   #                                      Identity   Status
-----------------------------------------------------------------
  79                       Momentum Anti-Palindrome      PASS
  80              Combined Time-Reversal Protection      PASS
  81                  Generation Protection Theorem      PASS
  82          Covering Tower Palindrome Persistence      PASS
  83             Profinite Spectral Self-Similarity      PASS
-----------------------------------------------------------------
Running total: 83 predictions/identities, 0 free parameters
Notebooks: 52 (NB01-NB52)
